In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. LOAD FACT TABLE
# ============================================================

df_fact = spark.table("workspace.sncf_gold.fact_stop_times")

print("===== FACT_STOP_TIMES =====")
print("Rows :", df_fact.count())

# ============================================================
# 2. EXECUTION PLAN
# Compatible avec Serverless
# ============================================================

df_fact.explain(mode="formatted")

===== FACT_STOP_TIMES =====
Rows : 8517376
== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet workspace.sncf_gold.fact_stop_times (1)


(1) PhotonScan parquet workspace.sncf_gold.fact_stop_times
Output [15]: [station_sk#29686, route_sk#29687, service_sk#29688, date_sk#29689, trip_id#29690, stop_id#29691, route_id#29692, service_id#29693, stop_sequence#29694, arrival_time#29695, departure_time#29696, pickup_type#29697, drop_off_type#29698, date#29699, _gold_processed_at#29700]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-zsugk/uc/f913ae46-a757-47d0-a4b0-7bdee9993ee0/0b25aeda-0cfe-485f-9891-a0a6f2ffbe7f/__unitystorage/catalogs/ab801c47-4962-468f-9614-a357ca04555a/tables/446ab0e5-937c-4646-a44c-6e46d198a7ba]
ReadSchema: struct<station_sk:int,route_sk:int,service_sk:int,date_sk:int,trip_id:string,stop_id:string,route_id:string,service_id:string,stop_sequence:int,arrival_time:string,departure_time:string,pickup_type:int,drop_off_type:int,d

In [0]:
print("===== DISTRIBUTION PAR DATE =====")

display(
    df_fact
    .groupBy("date_sk")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

===== DISTRIBUTION PAR DATE =====


date_sk,count
20260911,100031
20260904,99942
20261211,99716
20260925,99451
20261204,99434
20260918,99333
20261127,99185
20261002,99169
20261120,98810
20261106,98599


In [0]:
# ============================================================
# REPARTITION
# Redistribution complète des données
# Peut provoquer un shuffle
# ============================================================

df_repartitioned = (
    df_fact
    .repartition("date_sk")
)

df_repartitioned.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- == Initial Plan ==
   PhotonResultStage (6)
   +- PhotonColumnarToRow (5)
      +- PhotonShuffleExchangeSource (4)
         +- PhotonShuffleMapStage (3)
            +- PhotonShuffleExchangeSink (2)
               +- PhotonScan parquet workspace.sncf_gold.fact_stop_times (1)


(1) PhotonScan parquet workspace.sncf_gold.fact_stop_times
Output [15]: [station_sk#29820, route_sk#29821, service_sk#29822, date_sk#29823, trip_id#29824, stop_id#29825, route_id#29826, service_id#29827, stop_sequence#29828, arrival_time#29829, departure_time#29830, pickup_type#29831, drop_off_type#29832, date#29833, _gold_processed_at#29834]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-zsugk/uc/f913ae46-a757-47d0-a4b0-7bdee9993ee0/0b25aeda-0cfe-485f-9891-a0a6f2ffbe7f/__unitystorage/catalogs/ab801c47-4962-468f-9614-a357ca04555a/tables/446ab0e5-937c-4646-a44c-6e46d198a7ba]
ReadSchema: struct<station_sk:int,route_sk:int,service_sk:int,date_sk:int,trip_id:string,s

In [0]:
# ============================================================
# COALESCE
# Réduit généralement le nombre de partitions
# Évite un shuffle complet
# ============================================================

df_coalesced = df_fact.coalesce(4)

df_coalesced.explain(mode="formatted")

#repartition()
#→ peut augmenter ou réduire les partitions
#→ shuffle complet
#→ utile avant grosse opération distribuée

#coalesce()
#→ surtout réduire les partitions
#→ moins coûteux
#→ utile avant écriture si trop de petits fichiers

== Physical Plan ==
Coalesce (4)
+- * ColumnarToRow (3)
   +- PhotonResultStage (2)
      +- PhotonScan parquet workspace.sncf_gold.fact_stop_times (1)


(1) PhotonScan parquet workspace.sncf_gold.fact_stop_times
Output [15]: [station_sk#29857, route_sk#29858, service_sk#29859, date_sk#29860, trip_id#29861, stop_id#29862, route_id#29863, service_id#29864, stop_sequence#29865, arrival_time#29866, departure_time#29867, pickup_type#29868, drop_off_type#29869, date#29870, _gold_processed_at#29871]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-zsugk/uc/f913ae46-a757-47d0-a4b0-7bdee9993ee0/0b25aeda-0cfe-485f-9891-a0a6f2ffbe7f/__unitystorage/catalogs/ab801c47-4962-468f-9614-a357ca04555a/tables/446ab0e5-937c-4646-a44c-6e46d198a7ba]
ReadSchema: struct<station_sk:int,route_sk:int,service_sk:int,date_sk:int,trip_id:string,stop_id:string,route_id:string,service_id:string,stop_sequence:int,arrival_time:string,departure_time:string,pickup_type:int,drop_off_type:int,date:date,_gold_processed_

In [0]:
nb_dates = (
    df_fact
    .select("date_sk")
    .distinct()
    .count()
)

print("Nombre de dates distinctes :", nb_dates)
display(
    df_fact
    .groupBy("date_sk")
    .agg(
        F.count("*").alias("nb_rows")
    )
    .orderBy("date_sk")
)

 

Nombre de dates distinctes : 108


date_sk,nb_rows
20260903,97373
20260904,99942
20260905,61853
20260906,53044
20260907,98093
20260908,97485
20260909,97589
20260910,97658
20260911,100031
20260912,61454


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# ============================================================
# 1. RECHARGER LES TABLES
# ============================================================

stop_times = spark.table("workspace.sncf_silver.stop_times")

trips = (
    spark.table("workspace.sncf_silver.trips")
    .select("trip_id", "route_id", "service_id")
)

calendar_dates = (
    spark.table("workspace.sncf_silver.calendar_dates")
    .filter(F.col("exception_type") == 1)
    .select("service_id", "date")
)

dim_station = (
    spark.table("workspace.sncf_gold.dim_station")
    .select("stop_id", "station_sk")
)

dim_route = (
    spark.table("workspace.sncf_gold.dim_route")
    .select("route_id", "route_sk")
)

dim_service = (
    spark.table("workspace.sncf_gold.dim_service")
    .select("service_id", "service_sk")
)

dim_date = (
    spark.table("workspace.sncf_gold.dim_date")
    .select("date", "date_sk")
)

# ============================================================
# 2. JOIN OPTIMISÉ
# ============================================================

df_fact_optimized = (
    stop_times
    .join(trips, "trip_id", "inner")
    .join(broadcast(calendar_dates), "service_id", "inner")
    .join(broadcast(dim_station), "stop_id", "left")
    .join(broadcast(dim_route), "route_id", "left")
    .join(broadcast(dim_service), "service_id", "left")
    .join(broadcast(dim_date), "date", "left")
)

# ============================================================
# 3. VOIR LE PLAN D'EXÉCUTION
# ============================================================

df_fact_optimized.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (41)
+- == Initial Plan ==
   PhotonResultStage (40)
   +- PhotonColumnarToRow (39)
      +- PhotonProject (38)
         +- PhotonBroadcastHashJoin LeftOuter (37)
            :- PhotonProject (32)
            :  +- PhotonBroadcastHashJoin LeftOuter (31)
            :     :- PhotonProject (26)
            :     :  +- PhotonBroadcastHashJoin LeftOuter (25)
            :     :     :- PhotonProject (20)
            :     :     :  +- PhotonBroadcastHashJoin LeftOuter (19)
            :     :     :     :- PhotonProject (14)
            :     :     :     :  +- PhotonBroadcastHashJoin Inner (13)
            :     :     :     :     :- PhotonProject (7)
            :     :     :     :     :  +- PhotonBroadcastHashJoin Inner (6)
            :     :     :     :     :     :- PhotonScan parquet workspace.sncf_silver.stop_times (1)
            :     :     :     :     :     +- PhotonShuffleExchangeSource (5)
            :     :     :     :     :        +- PhotonSh

In [0]:
# ============================================================
# OPTIMISATION DELTA - FACT_STOP_TIMES
# ============================================================

spark.sql("""
    OPTIMIZE workspace.sncf_gold.fact_stop_times
""")

print("OPTIMIZE terminé")

OPTIMIZE terminé


In [0]:
# ============================================================
# STATISTIQUES POUR L'OPTIMIZER
# ============================================================

spark.sql("""
    ANALYZE TABLE workspace.sncf_gold.fact_stop_times
    COMPUTE STATISTICS
""")

print("Statistiques calculées")

Statistiques calculées
